# Подготовка данных, чанкование и построение векторного индекса для Enterprise Private GPT

Этот блокнот содержит пошаговый пайплайн обработки документов для корпоративной RAG-системы (Retrieval-Augmented Generation).

### Основные шаги:
1. **Загрузка данных:** Загрузка юридических или корпоративных текстов из датасета https://github.com/zeino8/RuLegalNER.
2. **Очистка текста (Preprocessing):** Удаление лишних пробелов, табуляций и управляющих символов для повышения качества эмбеддингов.
3. **Чанкование (Chunking):** Разделение очищенного текста на фрагменты фиксированного размера с перекрытием с помощью `RecursiveCharacterTextSplitter` из LangChain.
4. **Сохранение чанков:** Экспорт фрагментов в эффективный текстовый формат JSONL (для экономии ОЗУ и удобной потоковой работы).
5. **Векторизация (Embeddings):** Пакетная генерация эмбеддингов с использованием локальной модели `text-embedding-nomic-embed-text-v1.5` через локальный API (LM Studio / Ollama).
6. **Сохранение эмбеддингов:** Сохранение векторов в NumPy-массив (`.npy`).
7. **Векторный поиск (FAISS):** Построение индекса `FAISS (IndexFlatL2)` для быстрого семантического поиска наиболее релевантных фрагментов по запросу.

In [4]:
import pandas as pd
import json
from pathlib import Path
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
import requests
import numpy as np
import faiss
import json

# Путь для сохранения результирующих чанков в формате JSON Lines (каждая строка - отдельный JSON)
output_path = Path("chunks.jsonl")

# --- Конфигурационные параметры RAG-пайплайна ---

# Размер чанка в символах
CHUNK_SIZE = 1000
# Величина перекрытия чанков (в символах) для сохранения связности контекста на стыках
CHUNK_OVERLAP = 150

# Параметры локальных моделей и API
EMBEDDING_MODEL = "text-embedding-nomic-embed-text-v1.5"
LM_STUDIO_URL = "http://localhost:1234/v1"

# Размер пакета (батча) для оптимизации сетевых запросов при генерации эмбеддингов
BATCH_SIZE = 256

# --- Загрузка исходных документов для визуального анализа ---
# Читаем датасет, в котором содержатся тексты документов (например, юридических решений)
df = pd.read_csv(
    "../../dataset/test.csv",
    encoding="utf-8",
    header=None
)

# Вывод базовой информации о загруженной структуре данных
print(f"Документов: {len(df)}")
print(f"Столбцов: {len(df.columns)}")
print("\nФрагмент текста (первые 500 символов):")
print(df[0].iloc[0][:500])

Документов: 10000
Столбцов: 2

Фрагмент текста (первые 500 символов):


	       	Решение по административному делу
      	

Дело № 5-29/2013

ПОСТАНОВЛЕНИЕ

г. Биробиджан                                                                                                 <ДАТА1>

Мировой судья Центрального судебного участка города Биробиджана Еврейской автономной области Стасенко О.Н., 
рассмотрев материал об административном правонарушении в отношении Шевченко Андрея Витальевича, <ДАТА2> рождения, уроженца <АДРЕС>, 

УСТАНОВИЛ:

В 14 час. 35 минут <ДАТА3> в <АДРЕС>


## 1. Очистка и нормализация текста

Предобработка текста критически важна перед векторизацией. Функция `clean_text` выполняет следующие задачи:
- Приводит переносы строк Windows/Mac к стандартному виду (`\n`).
- Заменяет символы табуляции (`\t`) и неразрывные пробелы (`\u00a0`) на обычные пробелы.
- Схлопывает последовательности пробелов (2 и более) в один.
- Ограничивает избыточные пустые строки до максимум двух переносов подряд (`\n\n`).
- Удаляет пробельные символы на краях документа.

In [112]:
def clean_text(text):
    """
    Выполняет очистку и нормализацию сырого текста перед чанкованием.
    
    Параметры:
        text (str): Исходный необработанный текст документа.
        
    Возвращает:
        str: Очищенный и нормализованный текст без избыточных пробельных символов.
    """
    # Нормализуем переносы строк
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # Заменяем знаки табуляции на пробелы
    text = text.replace("\t", " ")
    # Заменяем неразрывные пробелы на стандартные
    text = text.replace("\u00a0", " ")
    # Схлопываем множественные пробелы в один
    text = re.sub(r"[ ]{2,}", " ", text)
    # Ограничиваем количество пустых строк максимум двумя подряд
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

## 2. Разделение текста на фрагменты (Чанкование / Chunking)

Мы используем `RecursiveCharacterTextSplitter` из LangChain. Данный сплиттер рекурсивно разделяет текст, ориентируясь на структуру абзацев, предложений и слов, что позволяет сохранить семантическую ценность каждого чанка.

- `chunk_size=1000` задает максимальный размер фрагмента.
- `chunk_overlap=150` обеспечивает плавный переход контекста между соседними чанками.

In [114]:
# Инициализация сплиттера из библиотеки LangChain
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

def split_text(text):
    """
    Разбивает подготовленный текст документа на чанки фиксированного размера с перекрытием.
    
    Параметры:
        text (str): Очищенный текст документа.
        
    Возвращает:
        list: Список текстовых фрагментов (чанков).
    """
    return splitter.split_text(text)


## 3. Обработка датасета документов

Проходим циклом по всем документам датасета, выполняем очистку и чанкование. Каждому чанку присваивается уникальный идентификатор вида `document_id_chunk_id` для возможности быстрого восстановления связи с исходным документом.

In [115]:
# Плоский список для хранения метаданных и текста всех сформированных чанков
all_chunks = []

# Итерируемся по документам из DataFrame
for document_id, text in df[0].items():
    # Пропускаем пустые строки
    if pd.isna(text):
        continue

    # Приводим к строке и очищаем текст от лишнего шума
    cleaned_text = clean_text(str(text))
    # Делим очищенный документ на фрагменты
    chunks_list = split_text(cleaned_text)

    # Формируем структуру чанка с метаданными
    for chunk_id, chunk in enumerate(chunks_list):
        all_chunks.append({
            "chunk_id": f"{document_id}_{chunk_id}",
            "document_id": document_id,
            "text": chunk
        })

print(f"Документов обработано: {df[0].notna().sum()}")
print(f"Всего чанков: {len(all_chunks)}")

Документов обработано: 10000
Всего чанков: 86772


## 4. Экспорт чанков в JSON Lines (JSONL)

Сохраняем обработанные чанки в файл `chunks.jsonl`. Формат JSONL является стандартным в инженерии данных для потоковой обработки, так как позволяет считывать документы построчно без загрузки всего файла целиком в ОЗУ.

In [116]:
# Открываем файл для записи с кодировкой UTF-8
with output_path.open("w", encoding="utf-8") as f:
    # Перебираем все созданные чанки
    for chunk in all_chunks:
        # Записываем JSON-объект в файл, сохраняя оригинальные символы (не экранируем кириллицу)
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")
        
# Выводим статистику о результатах записи
print(f"Сохранено чанков: {len(all_chunks)}")
print(f"Файл: {output_path}")

Сохранено чанков: 86772
Файл: chunks.jsonl


In [5]:
# Загружаем чанки из сохранённого файла для дальнейшего использования (восстановление состояния)
with open("chunks.jsonl", "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

print("Загружено чанков:", len(chunks))

Загружено чанков: 86772


## 5. Генерация семантических эмбеддингов

Каждому текстовому чанку мы сопоставляем числовой вектор размерности 768. 
Для генерации эмбеддингов используется локальный API (LM Studio), развернутый на `{LM_STUDIO_URL}/embeddings`.

- **Модель:** `text-embedding-nomic-embed-text-v1.5`
- **Метод:** Пакетный (батчи по 256 чанков) для уменьшения накладных расходов сети и эффективной утилизации GPU.

*Важно: Время выполнения данного шага для 86 000+ чанков на локальном компьютере может быть длительным, поэтому полученные эмбеддинги кэшируются в `.npy` файл.*

In [ ]:
# Список для накопления векторов-эмбеддингов
embeddings = []

# Генерация эмбеддингов батчами
for start in range(0, len(chunks), BATCH_SIZE):
    # Выделяем батч текстов чанков
    batch = [chunk["text"] for chunk in chunks[start:start + BATCH_SIZE]]

    # Отправляем POST-запрос к локальному API эмбеддингов
    response = requests.post(
        f"{LM_STUDIO_URL}/embeddings",
        json={
            "model": EMBEDDING_MODEL,
            "input": batch
        }
    )

    # Проверяем статус ответа (бросит исключение в случае ошибки сервера)
    response.raise_for_status()

    # Добавляем эмбеддинги в общий список
    data = response.json()
    embeddings.extend(item["embedding"] for item in data["data"])

    # Расчет прогресса обработки
    processed = min(start + BATCH_SIZE, len(chunks))

    # Периодически логируем прогресс каждые 2560 чанков
    if processed % 2560 == 0 or processed == len(chunks):
        print(f"Обработано: {processed} / {len(chunks)}")

print(f"Всего эмбеддингов: {len(embeddings)}")
print(f"Размерность: {len(embeddings[0]) if embeddings else 'Н/Д'}")

Обработано: 2560 / 86772
Обработано: 5120 / 86772
Обработано: 7680 / 86772
Обработано: 10240 / 86772
Обработано: 12800 / 86772
Обработано: 15360 / 86772
Обработано: 17920 / 86772
Обработано: 20480 / 86772
Обработано: 23040 / 86772
Обработано: 25600 / 86772
Обработано: 28160 / 86772
Обработано: 30720 / 86772
Обработано: 33280 / 86772
Обработано: 35840 / 86772
Обработано: 38400 / 86772
Обработано: 40960 / 86772
Обработано: 43520 / 86772
Обработано: 46080 / 86772
Обработано: 48640 / 86772
Обработано: 51200 / 86772
Обработано: 53760 / 86772
Обработано: 56320 / 86772
Обработано: 58880 / 86772
Обработано: 61440 / 86772
Обработано: 64000 / 86772
Обработано: 66560 / 86772
Обработано: 69120 / 86772
Обработано: 71680 / 86772
Обработано: 74240 / 86772
Обработано: 76800 / 86772
Обработано: 79360 / 86772
Обработано: 81920 / 86772
Обработано: 84480 / 86772
Обработано: 86772 / 86772
Всего эмбеддингов: 86772
Размерность: 768


## 6. Кэширование векторов эмбеддингов в NumPy

Для того чтобы не генерировать векторы заново при перезапуске сессии, мы приводим их к типу данных `float32` (требование FAISS) и сохраняем на диск в виде бинарного NumPy-массива `.npy`.

In [ ]:
# Преобразуем список списков в NumPy-массив с типом float32 (оптимальный формат для поиска)
embeddings_array = np.array(embeddings, dtype="float32")

# Сохраняем массив на диск
np.save("embeddings.npy", embeddings_array)

# Контрольный вывод формы полученной матрицы
print(embeddings_array.shape)

## 7. Создание и наполнение векторного индекса FAISS

**FAISS (Facebook AI Similarity Search)** — библиотека для быстрого семантического поиска в больших коллекциях векторов.
- Мы используем индекс `IndexFlatL2`, который выполняет точный поиск (Exact Search) на основе евклидова расстояния (L2-норма).
- Этот индекс выступает в качестве легковесного, высокоскоростного локального векторного хранилища.

In [7]:
# Загружаем сохранённые эмбеддинги
embeddings_array = np.load("embeddings.npy")

# Создаём FAISS-индекс для векторов заданной размерности (в нашем случае - 768) 
index = faiss.IndexFlatL2(embeddings_array.shape[1])


# Добавляем все эмбеддинги в поисковый индекс
index.add(embeddings_array)

# Проверяем количество добавленных векторов и их размерность
print("Количество векторов:", index.ntotal)
print("Размерность:", index.d)

# Сохраняем FAISS-индекс на диск для быстрой загрузки в будущем
faiss.write_index(index, "faiss.index")

print("FAISS-индекс сохранён: faiss.index")

# Загружаем сохранённый FAISS-индекс (демонстрация восстановления)
index = faiss.read_index("faiss.index")

Количество векторов: 86772
Размерность: 768
FAISS-индекс сохранён: faiss.index
